<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/Continuous_Bag_of_Words.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import urllib.request

In [2]:
#Dataset
url = "https://www.gutenberg.org/cache/epub/84/pg84.txt"
output_file = "/content/frankenstein.txt"
urllib.request.urlretrieve(url, output_file)

print("Downloaded to:", output_file)

Downloaded to: /content/frankenstein.txt


In [3]:
print(os.path.getsize("/content/frankenstein.txt"))

448885


In [4]:
import os
from argparse import Namespace
import collections
import nltk.data #Natural Language Toolkit (NLTK) library used to find, load, and manage external datasets, pre-trained models, and corpora in Python
import numpy as np
import pandas as pd
import re
from tqdm import tqdm_notebook

In [5]:
args = Namespace(
    raw_dataset_txt="/content/frankenstein.txt",
    train_proportion=0.7,
    val_proportion=0.15,
    test_proportion=0.15,
    window_size = 5,
    output_munged_csv="/content/frankenstein_with_splits.csv",
    seed=1337
)

In [6]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
with open(args.raw_dataset_txt) as fp:
  book = fp.read()
sentences = tokenizer.tokenize(book)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
print(len(sentences))
print(sentences[:30])

3199
['The Project Gutenberg eBook of Frankenstein; or, the modern prometheus\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever.', 'You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this eBook or online\nat www.gutenberg.org.', 'If you are not located in the United States,\nyou will have to check the laws of the country where you are located\nbefore using this eBook.', 'Title: Frankenstein; or, the modern prometheus\n\nAuthor: Mary Wollstonecraft Shelley\n\n\n        \nRelease date: October 1, 1993 [eBook #84]\n                Most recently updated: February 10, 2026\n\nLanguage: English\n\nOther information and formats: www.gutenberg.org/ebooks/84\n\nCredits: Judith Boss, Christy Phillips, Lynn Hanninen and David Meltzer.', 'HTML version by Al Haines.', 'Further corrections by Menno de Leeuw.', '*** START OF THE PROJE

In [8]:
#Clean Sentences
def preprocess_text(text):
  text = ' '.join(word.lower() for word in text.split(" "))
  text = re.sub(r"([.,!?])", r" \1 ", text)
  text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
  return text

In [9]:
cleaned_sentences = [preprocess_text(sentence) for sentence in sentences]

In [10]:
MASK_TOKEN = '<MASK>'

In [11]:
#Create Windows
flatten = lambda outer_list: [item for inner_list in outer_list for item in inner_list] #Sqeeze all arrays to 1D
#Slidding window with <MASK>'s as padding
windows = flatten([list(nltk.ngrams([MASK_TOKEN]*args.window_size + sentence.split(' ') + [MASK_TOKEN]* \
                                    args.window_size,args.window_size*2+1)) for sentence in tqdm_notebook(cleaned_sentences)])

/tmp/ipykernel_374/759950361.py:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  args.window_size,args.window_size*2+1)) for sentence in tqdm_notebook(cleaned_sentences)])


  0%|          | 0/3199 [00:00<?, ?it/s]

In [12]:
#Create CBOW Data
data =[]
for window in tqdm_notebook(windows):
  target_token = window[args.window_size]
  context =[]
  for i,token in enumerate (window):
    if token==MASK_TOKEN or i==args.window_size:
      continue
    else:
      context.append(token)
  data.append([' '.join(token for token in context),target_token])

/tmp/ipykernel_374/2067953158.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for window in tqdm_notebook(windows):


  0%|          | 0/90578 [00:00<?, ?it/s]

In [13]:
cbow_data = pd.DataFrame(data,columns=["context","target"])

In [14]:
cbow_data.head()

,context,target
0,project gutenberg ebook of frankenstein,the
1,the gutenberg ebook of frankenstein or,project
2,"the project ebook of frankenstein or ,",gutenberg
3,"the project gutenberg of frankenstein or , the",ebook
4,"the project gutenberg ebook frankenstein or , ...",of


In [15]:
#Split the data
n=len(cbow_data)
def get_split(row_num):
  if row_num <= n*args.train_proportion:
    return 'train'
  elif (row_num > n*args.train_proportion) and (row_num <= n*args.train_proportion + n*args.val_proportion):
    return 'val'
  else:
    return 'test'

cbow_data['split'] = cbow_data.apply(lambda row: get_split(row.name),axis=1)

In [16]:
cbow_data.head()

,context,target,split
0,project gutenberg ebook of frankenstein,the,train
1,the gutenberg ebook of frankenstein or,project,train
2,"the project ebook of frankenstein or ,",gutenberg,train
3,"the project gutenberg of frankenstein or , the",ebook,train
4,"the project gutenberg ebook frankenstein or , ...",of,train


In [17]:
cbow_data.split.value_counts()

,count
split,
train,63405
val,13587
test,13586


In [18]:
cbow_data.to_csv(args.output_munged_csv, index=False)

Learning Embeddings with Continuous Bag of Words (CBOW)

In [19]:
import os
from argparse import Namespace
from collections import Counter
import json
import re
import string

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm_notebook

In [82]:
class Vocabulary(object):
  def __init__(self,token_to_idx=None,mask_token='<MASK>',add_unk=True,unk_token='<UNK>'):
    if token_to_idx is None:
      token_to_idx = {}
    self._token_to_idx = token_to_idx
    self._idx_to_token = {idx:token for token,idx in self._token_to_idx.items()}
    self._add_unk = add_unk
    self._unk_token = unk_token
    self._mask_token = mask_token
    self.mask_index = self.add_token(self._mask_token)
    if add_unk:
      self.unk_index = self.add_token(unk_token)

  def to_serializable(self):
    return {'token_to_idx':self._token_to_idx,
            'add_unk':self._add_unk,
            'unk_token':self._unk_token,
            'mask_token':self._mask_token}

  @classmethod
  def from_serializable(cls,contents):
    return cls(**contents)

  def add_token(self,token):
    if token in self._token_to_idx:
      index = self._token_to_idx[token]
    else:
      index = len(self._token_to_idx)
      self._token_to_idx[token] = index
      self._idx_to_token[index] = token
    return index

  def add_many(self,tokens):
    return [self.add_token(token) for token in tokens]

  def lookup_token(self,token):
    if self.unk_index>=0:
      return self._token_to_idx.get(token,self.unk_index)
    else:
      return self._token_to_idx[token]

  def lookup_index(self,index):
    if index not in self._idx_to_token:
      raise KeyError('The index not in vocab')
    return self._idx_to_token[index]

  def __str__(self):
    return f'<Vocabulary(size={len(self)})>'

  def __len__(self):
    return len(self._token_to_idx)

  def __str__(self) -> str:
    return f'<Vocabulary(size={len(self)})>'

  def __len__(self) -> int:
    return len(self._token_to_idx)

In [90]:
class CWOBVectorizer(object):
  def __init__(self,cbow_vocab):
    self.cbow_vocab = cbow_vocab

  def vectorize(self,context,vector_lenght=-1):
    indices = [self.cbow_vocab.lookup_token(token) for token in context.split(' ')]
    if vector_lenght < 0:
      vector_lenght = len(indices)
    out_vector = np.zeros(vector_lenght,dtype=np.int64)
    out_vector[:len(indices)] = indices
    out_vector[len(indices):]=self.cbow_vocab.mask_index
    return out_vector

  @classmethod
  def from_dataframe(cls,cbow_df):
    cbow_vocab = Vocabulary()
    for index,row in cbow_df.iterrows():
      for token in row.context.split(' '):
        cbow_vocab.add_token(token)
      cbow_vocab.add_token(row.target)
    return cls(cbow_vocab)

  @classmethod
  def from_serializable(cls,contents):
    cbow_vocab = Vocabulary.from_serializable(contents['cbow_vocab'])
    return cls(cbow_vocab=cbow_vocab)

  def to_serializable(self):
    return {'cbow_vocab':self.cbow_vocab.to_serializable()}

In [84]:
class CBOWDataset(Dataset):
  def __init__(self,cbow_df,vectorizer):
    self.cbow_df = cbow_df
    self._vectorizer = vectorizer

    measure_len = lambda context:len(context.split(' '))
    self._max_seq_length = max(map(measure_len,cbow_df.context))

    self.train_df = self.cbow_df[self.cbow_df.split=='train']
    self.train_size = len(self.train_df)

    self.val_df = self.cbow_df[self.cbow_df.split=='val']
    self.validation_size = len(self.val_df)

    self.test_df = self.cbow_df[self.cbow_df.split=='test']
    self.test_size = len(self.test_df)

    self._lookup_dict = {'train':(self.train_df,self.train_size),
                         'val':(self.val_df,self.validation_size),
                         'test':(self.test_df,self.test_size)}
    self.set_split('train')

  @classmethod
  def load_dataset_make_vectorizer(cls,cbow_csv):
    cbow_df = pd.read_csv(cbow_csv)
    train_cbow_df = cbow_df[cbow_df.split=='train']
    return cls(cbow_df,CWOBVectorizer.from_dataframe(train_cbow_df))

  @classmethod
  def load_dataset_and_load_vectorizer(cls,cbow_csv,vectorizer_filepath):
    cbow_df = pd.read_csv(cbow_csv)
    vectorizer = cls.load_vectorizer_only(vectorizer_filepath)
    return cls(cbow_df,vectorizer)

  @staticmethod
  def load_vectorizer_only(vectorizer_filepath):
    with open(vectorizer_filepath) as fp:
      return CWOBVectorizer.from_serializable(json.load(fp))

  def save_vectorizer(self,vectorizer_filepath):
    with open(vectorizer_filepath,'w') as fp:
      json.dump(self._vectorizer.to_serializable(),fp)

  def get_vectorizer(self):
    return self._vectorizer

  def set_split(self,split='train'):
    self._target_split = split
    self._target_df,self._target_size = self._lookup_dict[split]

  def __len__(self):
    return self._target_size

  def __getitem__(self, index):
    row = self._target_df.iloc[index]
    context_vector = self._vectorizer.vectorize(row.context,self._max_seq_length)
    target_index = self._vectorizer.cbow_vocab.lookup_token(row.target)
    return {'x_data':context_vector,'y_target':target_index}

  def get_num_batches(self,batch_size):
    return len(self)//batch_size


In [61]:
def generate_batches(dataset,batch_size,shuffle=True,drop_last=True,device='cpu'):
  dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  for data_dict in dataloader:
    out_data_dict = {}
    for name,tensor in data_dict.items():
      out_data_dict[name]=data_dict[name].to(device)
    yield out_data_dict

In [62]:
class CBOWClassifer(nn.Module):
  def __init__(self,vocabulary_size,embedding_size,padding_idx):
    super(CBOWClassifer,self).__init__()
    self.embedding = nn.Embedding(num_embeddings=vocabulary_size,embedding_dim=embedding_size,padding_idx=padding_idx)
    self.fc1 = nn.Linear(in_features = embedding_size,out_features=vocabulary_size)

  def forward(self,x_in,apply_softmax=False):
    x_embedded_sum = F.dropout(self.embedding(x_in).sum(dim=1),0.3)
    y_out = self.fc1(x_embedded_sum)
    if apply_softmax:
      y_out = F.softmax(y_out,dim=1)
    return y_out

In [63]:
def make_train_state(args):
    return {'stop_early': False,
            'early_stopping_step': 0,
            'early_stopping_best_val': 1e8,
            'learning_rate': args.learning_rate,
            'epoch_index': 0,
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'test_loss': -1,
            'test_acc': -1,
            'model_filename': args.model_state_file}

In [64]:
def update_train_state(args, model, train_state):
   # Save one model at least
    if train_state['epoch_index'] == 0:
        torch.save(model.state_dict(), train_state['model_filename'])
        train_state['stop_early'] = False

    # Save model if performance improved
    elif train_state['epoch_index'] >= 1:
        loss_tm1, loss_t = train_state['val_loss'][-2:]

        # If loss worsened
        if loss_t >= train_state['early_stopping_best_val']:
            # Update step
            train_state['early_stopping_step'] += 1
        # Loss decreased
        else:
            # Save the best model
            if loss_t < train_state['early_stopping_best_val']:
                torch.save(model.state_dict(), train_state['model_filename'])

            # Reset early stopping step
            train_state['early_stopping_step'] = 0

        # Stop early ?
        train_state['stop_early'] = \
            train_state['early_stopping_step'] >= args.early_stopping_criteria

    return train_state

In [65]:
def compute_accuracy(y_pred, y_target):
    _, y_pred_indices = y_pred.max(dim=1)
    n_correct = torch.eq(y_pred_indices, y_target).sum().item()
    return n_correct / len(y_pred_indices) * 100

In [66]:
def set_seed_everywhere(seed, cuda):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if cuda:
        torch.cuda.manual_seed_all(seed)

def handle_dirs(dirpath):
    if not os.path.exists(dirpath):
        os.makedirs(dirpath)

In [67]:
args = Namespace(
    # Data and Path information
    cbow_csv="/content/frankenstein_with_splits.csv",
    vectorizer_file="vectorizer.json",
    model_state_file="model.pth",
    save_dir="/content/",
    # Model hyper parameters
    embedding_size=50,
    # Training hyper parameters
    seed=1337,
    num_epochs=100,
    learning_rate=0.0001,
    batch_size=32,
    early_stopping_criteria=5,
    # Runtime options
    cuda=True,
    catch_keyboard_interrupt=True,
    reload_from_files=False,
    expand_filepaths_to_save_dir=True
)

In [68]:
if args.expand_filepaths_to_save_dir:
    args.vectorizer_file = os.path.join(args.save_dir,
                                        args.vectorizer_file)

    args.model_state_file = os.path.join(args.save_dir,
                                         args.model_state_file)

    print("Expanded filepaths: ")
    print("\t{}".format(args.vectorizer_file))
    print("\t{}".format(args.model_state_file))

Expanded filepaths: 
	/content/vectorizer.json
	/content/model.pth


In [69]:
# Check CUDA
if not torch.cuda.is_available():
    args.cuda = False

args.device = torch.device("cuda" if args.cuda else "cpu")
print("Using CUDA: {}".format(args.cuda))

Using CUDA: True


In [70]:
set_seed_everywhere(args.seed, args.cuda)
handle_dirs(args.save_dir)

In [91]:
if args.reload_from_files:
    print("Loading dataset and loading vectorizer")
    dataset = CBOWDataset.load_dataset_and_load_vectorizer(args.cbow_csv,
                                                           args.vectorizer_file)
else:
    print("Loading dataset and creating vectorizer")
    dataset = CBOWDataset.load_dataset_make_vectorizer(args.cbow_csv)
    dataset.save_vectorizer(args.vectorizer_file)

Loading dataset and creating vectorizer


In [92]:
vectorizer = dataset.get_vectorizer()
classifier = CBOWClassifer(vocabulary_size=len(vectorizer.cbow_vocab),
                            embedding_size=args.embedding_size,
                            padding_idx=0)

In [93]:
classifier = classifier.to(args.device)
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(),lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,mode='min',factor=0.5,patience=1)

In [94]:
train_state = make_train_state(args)

In [96]:
try:
  for epoch_index in range(args.num_epochs):
    dataset.set_split('train')
    classifier.train()
    batch_generator = generate_batches(dataset,batch_size=args.batch_size,device=args.device)
    loss=0.0
    acc=0.0
    for batch_index,batch_dict in enumerate(batch_generator):
      optimizer.zero_grad()
      y_pred = classifier(x_in=batch_dict['x_data'])
      loss = loss_func(y_pred,batch_dict['y_target'])
      loss_batch = loss.item()
      loss.backward()
      optimizer.step()
      acc_batch = compute_accuracy(y_pred,batch_dict['y_target'])
      loss+=(loss_batch-loss)/(batch_index+1)
      acc+=(acc_batch-acc)/(batch_index+1)
    train_state['train_loss'].append(loss)
    train_state['train_acc'].append(acc)

    dataset.set_split('val')
    batch_generator = generate_batches(dataset,batch_size=args.batch_size,device=args.device)
    loss=0.0
    acc=0.0
    classifier.eval()
    with torch.no_grad():
      for batch_index,batch_dict in enumerate(batch_generator):
        y_pred = classifier(x_in=batch_dict['x_data'])
        loss = loss_func(y_pred,batch_dict['y_target'])
        loss_batch = loss.item()
        acc_batch = compute_accuracy(y_pred,batch_dict['y_target'])
        loss+=(loss_batch-loss)/(batch_index+1)
        acc+=(acc_batch-acc)/(batch_index+1)
      train_state['val_acc'].append(acc)
      train_state['val_loss'].append(loss)

    train_state = update_train_state(args=args,model=classifier,train_state=train_state)
    scheduler.step(train_state['val_loss'][-1])
    if train_state['stop_early']:
      break
    print("Epoch: {}/{}...".format(epoch_index+1,args.num_epochs),
          "Train Loss: {:.3f}".format(train_state['train_loss'][-1]),
          "Train Acc: {:.2f}".format(train_state['train_acc'][-1]),
          "Val Loss: {:.3f}".format(train_state['val_loss'][-1]),
          "Val Acc: {:.2f}".format(train_state['val_acc'][-1]))


except KeyboardInterrupt:
  print('Exiting')

Epoch: 1/100... Train Loss: 8.270 Train Acc: 5.21 Val Loss: 6.950 Val Acc: 6.14
Epoch: 2/100... Train Loss: 7.711 Train Acc: 7.25 Val Loss: 7.681 Val Acc: 7.86
Epoch: 3/100... Train Loss: 7.466 Train Acc: 8.55 Val Loss: 7.292 Val Acc: 9.20
Epoch: 4/100... Train Loss: 5.230 Train Acc: 9.25 Val Loss: 7.224 Val Acc: 9.07
Epoch: 5/100... Train Loss: 7.606 Train Acc: 9.70 Val Loss: 6.649 Val Acc: 9.68
Epoch: 6/100... Train Loss: 6.643 Train Acc: 10.12 Val Loss: 7.860 Val Acc: 10.03
Epoch: 7/100... Train Loss: 5.736 Train Acc: 10.45 Val Loss: 7.633 Val Acc: 10.28
Epoch: 8/100... Train Loss: 6.411 Train Acc: 10.84 Val Loss: 7.017 Val Acc: 10.61
Epoch: 9/100... Train Loss: 5.804 Train Acc: 10.88 Val Loss: 6.985 Val Acc: 10.58
Epoch: 10/100... Train Loss: 5.926 Train Acc: 10.98 Val Loss: 6.380 Val Acc: 10.39
Epoch: 11/100... Train Loss: 6.293 Train Acc: 10.94 Val Loss: 8.383 Val Acc: 10.69
Epoch: 12/100... Train Loss: 6.025 Train Acc: 11.06 Val Loss: 6.813 Val Acc: 10.76
Epoch: 13/100... Train 

In [101]:
dataset.set_split('test')
batch_generator = generate_batches(dataset,
                                   batch_size=args.batch_size,
                                   device=args.device)
running_loss = 0.0
running_acc = 0.0
classifier.eval()
with torch.no_grad():
    for batch_index,batch_dict in enumerate(batch_generator):
      y_pred = classifier(x_in=batch_dict['x_data'])
      loss = loss_func(y_pred,batch_dict['y_target'])
      loss_batch = loss.item()
      acc_batch = compute_accuracy(y_pred,batch_dict['y_target'])
      loss+=(loss_batch-loss)/(batch_index+1)
      acc+=(acc_batch-acc)/(batch_index+1)
train_state['test_loss'] = loss
train_state['test_acc'] = acc
print("Test loss: {};".format(train_state['test_loss']))
print("Test Accuracy: {}".format(train_state['test_acc']))

Test loss: 8.674707412719727;
Test Accuracy: 10.88590801886792


Trained Embeddings

In [97]:
def pretty_print(results):
  for item in results:
        print ("...[%.2f] - %s"%(item[1], item[0]))

def get_closest(target_word, word_to_idx, embeddings, n=5):
  word_embedding = embeddings[word_to_idx[target_word.lower()]]
  distances = []
  for word, index in word_to_idx.items():
      if word == "<MASK>" or word == target_word:
          continue
      distances.append((word, torch.dist(word_embedding, embeddings[index])))

  results = sorted(distances, key=lambda x: x[1])[1:n+2]
  return results

In [98]:
word = input('Enter a word: ')
embeddings = classifier.embedding.weight.data
word_to_idx = vectorizer.cbow_vocab._token_to_idx
pretty_print(get_closest(word, word_to_idx, embeddings, n=5))

Enter a word: monster
...[6.59] - recess
...[6.72] - obliged
...[6.93] - ensure
...[6.93] - continuing
...[6.94] - william
...[6.94] - afforded
